# Credit Risks 

Giving the risk in the credit card business, our role today will be to help the bank make better decision

## Expected Output 

For this project we want :
* Multiple accuracy measures resembling k-neignbors used for traimomg your KNN classifier
* One printed confusion matrix for the best model.



In [1]:
# import the dataset from Scikit-learn 

from sklearn.datasets import fetch_openml
import pandas as pd

# Load the credit dataset
credit = fetch_openml("credit-g", version=1, as_frame= True)

x = credit.data
y = credit.target

df = credit.data
df.head()


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   int64   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   int64   
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   int64   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   int64   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   int64   
 13  other_payment_plans     1000 non-null   category
 14  housing                 1000 non-nul

We have 20 columns we can decide to use all of them or choose some relevents columns 

For this project we are going to use 4 numeric features and 3 nominal features.

- Duration : longer the loan duration -> higher the risk
- credit_amount : Larger loans -> higher probability of default.
- installment_commitment : higher installment burden -> more stress on borrower -> higher risk
- age : Younger borrowers tend to have higher defaults rates

- checking_status : indicates liquidity and financial discipline
- credit_history : past repayment behaviours is the #1 predictor of future default.
- purpose : reason of the loan 

In [3]:
df = df[['checking_status', 'duration','credit_history', 'purpose','credit_amount','installment_commitment','age' ]]
df

,checking_status,duration,credit_history,purpose,credit_amount,installment_commitment,age
0,<0,6,critical/other existing credit,radio/tv,1169,4,67
1,0<=X<200,48,existing paid,radio/tv,5951,2,22
2,no checking,12,critical/other existing credit,education,2096,2,49
3,<0,42,existing paid,furniture/equipment,7882,2,45
4,<0,24,delayed previously,new car,4870,3,53
...,...,...,...,...,...,...,...
995,no checking,12,existing paid,furniture/equipment,1736,3,31
996,<0,30,existing paid,used car,3857,4,40
997,no checking,12,existing paid,radio/tv,804,4,38
998,<0,45,existing paid,radio/tv,1845,4,23


The table above is the final dataset.

checking for NaN value

In [4]:
df.isna().sum()

checking_status           0
duration                  0
credit_history            0
purpose                   0
credit_amount             0
installment_commitment    0
age                       0
dtype: int64

We don't hae any NaN value in the table.
now we do some pre-processing the dataset with the following code.

In [5]:
df['checking_status'].unique()

['<0', '0<=X<200', 'no checking', '>=200']
Categories (4, str): ['0<=X<200', '<0', '>=200', 'no checking']

In [ ]:
# Checking_status 
column_map = {
    'no checking': 0,
    '<0' : 1,
    '0<=X<200' : 2,
    '>=200' : 3,
}
df['checking_status'] = df['checking_status'].map(column_map)
df

,checking_status,duration,credit_history,purpose,credit_amount,installment_commitment,age
0,1,6,critical/other existing credit,radio/tv,1169,4,67
1,2,48,existing paid,radio/tv,5951,2,22
2,0,12,critical/other existing credit,education,2096,2,49
3,1,42,existing paid,furniture/equipment,7882,2,45
4,1,24,delayed previously,new car,4870,3,53
...,...,...,...,...,...,...,...
995,0,12,existing paid,furniture/equipment,1736,3,31
996,1,30,existing paid,used car,3857,4,40
997,0,12,existing paid,radio/tv,804,4,38
998,1,45,existing paid,radio/tv,1845,4,23


In [7]:
df['credit_history'].unique()

['critical/other existing credit', 'existing paid', 'delayed previously', 'no credits/all paid', 'all paid']
Categories (5, str): ['all paid', 'critical/other existing credit', 'delayed previously', 'existing paid', 'no credits/all paid']

In [ ]:
# Credit_history
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder()
credit_history = encoder.fit_transform(df[['credit_history']]).todense()
credit_history = pd.DataFrame(credit_history, columns=encoder.get_feature_names_out())
df.drop('credit_history', axis=1 , inplace=True)
df = pd.concat([df, credit_history], axis=1 )


In [9]:
df

,checking_status,duration,purpose,credit_amount,installment_commitment,age,credit_history_all paid,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid
0,1,6,radio/tv,1169,4,67,0.0,1.0,0.0,0.0,0.0
1,2,48,radio/tv,5951,2,22,0.0,0.0,0.0,1.0,0.0
2,0,12,education,2096,2,49,0.0,1.0,0.0,0.0,0.0
3,1,42,furniture/equipment,7882,2,45,0.0,0.0,0.0,1.0,0.0
4,1,24,new car,4870,3,53,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
995,0,12,furniture/equipment,1736,3,31,0.0,0.0,0.0,1.0,0.0
996,1,30,used car,3857,4,40,0.0,0.0,0.0,1.0,0.0
997,0,12,radio/tv,804,4,38,0.0,0.0,0.0,1.0,0.0
998,1,45,radio/tv,1845,4,23,0.0,0.0,0.0,1.0,0.0


In [ ]:
# Purpose
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder()
credit_history = encoder.fit_transform(df[['purpose']]).todense()
credit_history = pd.DataFrame(credit_history, columns=encoder.get_feature_names_out())
df.drop('purpose', axis=1 , inplace=True)
df = pd.concat([df, credit_history], axis=1 )

In [11]:
df

,checking_status,duration,credit_amount,installment_commitment,age,credit_history_all paid,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid,purpose_business,purpose_domestic appliance,purpose_education,purpose_furniture/equipment,purpose_new car,purpose_other,purpose_radio/tv,purpose_repairs,purpose_retraining,purpose_used car
0,1,6,1169,4,67,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,48,5951,2,22,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0,12,2096,2,49,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,42,7882,2,45,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,24,4870,3,53,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,12,1736,3,31,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
996,1,30,3857,4,40,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
997,0,12,804,4,38,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
998,1,45,1845,4,23,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
# class 
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [13]:
df

,checking_status,duration,credit_amount,installment_commitment,age,credit_history_all paid,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid,purpose_business,purpose_domestic appliance,purpose_education,purpose_furniture/equipment,purpose_new car,purpose_other,purpose_radio/tv,purpose_repairs,purpose_retraining,purpose_used car
0,1,6,1169,4,67,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,48,5951,2,22,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0,12,2096,2,49,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,42,7882,2,45,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,24,4870,3,53,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,12,1736,3,31,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
996,1,30,3857,4,40,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
997,0,12,804,4,38,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
998,1,45,1845,4,23,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
# Now we have our dataset let check the shape x and y most have the same shape
df.shape, y.shape

((1000, 20), (1000,))

In [ ]:
# we split our data into train and test
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(df,y, test_size=0.2, random_state=0)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 20), (200, 20), (800,), (200,))

In [ ]:
# we standarlize the data to be on the same scale 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# we fit the data to the KNeighbors classifier for training 
from sklearn.neighbors import KNeighborsClassifier 
# Choose value of K 
K = 64
model = KNeighborsClassifier(n_neighbors=K)
model.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",64
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [ ]:
# We find the prediction rate 
y_pred = model.predict(X_test)

In [ ]:
# We get the overall score of our model
y_score= model.score(X_test, y_test)
y_score

0.715

In [ ]:
# We find the accuracy witch is the way to know how well our model works
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.715

In [ ]:
# Same but with a different way to present the result.
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[  6  52]
 [  5 137]]
              precision    recall  f1-score   support

           0       0.55      0.10      0.17        58
           1       0.72      0.96      0.83       142

    accuracy                           0.71       200
   macro avg       0.64      0.53      0.50       200
weighted avg       0.67      0.71      0.64       200

